# 🔍 Selecting rows and columns

**Time:** ~25 minutes &nbsp;•&nbsp; **Goal:** stop looking at the whole table and start **asking it things** — exactly the rows you want, in one line.

This is the notebook that pays off most often. Almost every analysis you write is a selection followed by a summary, and the selection is where the mistakes live: the wrong brackets, the wrong `and`, a date column that is secretly text.

## Run this first

In [ ]:
# Run this first. Same file as notebook 1, with the header bug already fixed.
import urllib.request
from pathlib import Path

import pandas as pd

DATA_URL = "https://raw.githubusercontent.com/HocheggerLab/y3-bio-python/main/lab06/data/"

SURVEY = "pollinators.csv"
if not Path(SURVEY).exists():
    urllib.request.urlretrieve(DATA_URL + SURVEY, SURVEY)

df = pd.read_csv(SURVEY)
df.columns = df.columns.str.strip()

INSECTS = ["Beetles", "Hoverflies", "Flies", "Butterflies",
           "Honeybees", "Solitary_Bees", "Wasps", "Bumblebees"]

print(df.shape)
df.head(3)

## Skill 1 — `.loc` and `.iloc`

`df["Honeybees"]` gets a column. For **rows**, or rows and columns together, you need one of these:

| | selects by | example |
|---|---|---|
| `.loc` | **label** — the index value and the column name | `df.loc[5, "Honeybees"]` |
| `.iloc` | **integer position** — like a list | `df.iloc[5, 13]` |

Right now the index is `0, 1, 2, …`, so the two look almost the same. They stop looking the same the moment you sort or filter, because **the index travels with the row**. That is the point of it.

One gotcha: `.loc` slices are **inclusive** of the end label, unlike everything else in Python.

In [ ]:
# DEMO - label vs position
print("--- one row, by label ---")
print(df.loc[5, ["Date", "Allotment", "Crop_Generic", "Bumblebees"]])

print()
print("--- .loc is inclusive: 5 to 8 gives FOUR rows ---")
print(df.loc[5:8, ["Allotment", "Bumblebees"]])

print()
print("--- .iloc is exclusive, like a list slice: three rows ---")
print(df.iloc[5:8, :3])

In [ ]:
# DEMO - why the index matters
top = df.nlargest(3, "Bumblebees")
print(top[["Allotment", "Crop_Generic", "Bumblebees"]])

print()
print("iloc[0] -> the first row of the SORTED table:")
print(top.iloc[0][["Allotment", "Bumblebees"]].to_dict())

print()
print("the labels that came along for the ride:", top.index.tolist())

### 🔍 DIY challenge

1. Print rows 100 to 104 with `.loc`, showing only `Date`, `Crop_Generic` and `Honeybees`. How many rows did you get?
2. Get the same **shape** of answer with `.iloc`. What did you have to change?
3. `df.iloc[-1]` gives the last row. Is there a `.loc` equivalent? Try `df.loc[len(df) - 1]` and say why it happens to work here.

> 💡 If you find yourself counting columns to work out a number for `.iloc`, you wanted `.loc`.

In [ ]:
# TODO - your turn

## Skill 2 — A question, asked of every row

A comparison on a column gives a Series of `True`/`False` — one per row. This is the **boolean mask** you met with numpy, and it does the same job here: `mask.sum()` counts the `True`s, `mask.mean()` gives you the proportion.

To combine conditions, pandas uses `&` (and), `|` (or), `~` (not) — **not** `and`, `or`, `not`, which insist on a single `True`/`False`. And because `&` binds more tightly than `>`, **every condition needs its own brackets**:

```python
df[(df["Bumblebees"] > 0) & (df["Year"] == 2018)]   # right
df[df["Bumblebees"] > 0 & df["Year"] == 2018]       # raises
```

This is the most common error in pandas, and you will make it today.

In [ ]:
# DEMO - how much of this survey is zeros?
seen = df["Bumblebees"] > 0

print("watches with at least one bumblebee:", seen.sum(), "of", len(df))
print("as a proportion:", round(seen.mean(), 3))

print()
bb = df[seen]
print("when bumblebees WERE seen, the mean count was", round(bb["Bumblebees"].mean(), 2))
print("but across all watches it is only        ", round(df["Bumblebees"].mean(), 2))

That gap between the two numbers is not a detail — it is the shape of the data. Most watches saw nothing. Any mean you calculate later is a mean over a lot of zeros, and you will need to decide, every time, whether that is what you want.

In [ ]:
# DEMO - bumblebees on runner beans, in 2018 only
mask = (df["Crop_Generic"] == "Rbean") & (df["Year"] == 2018) & (df["Bumblebees"] > 0)
print(mask.sum(), "matching watches")

df[mask][["Date", "Allotment", "Total_Flowers", "Bumblebees"]].head()

In [ ]:
# DEMO - what the wrong version actually does
try:
    df[df["Bumblebees"] > 0 & df["Year"] == 2018]
except Exception as e:
    print(type(e).__name__)
    print(e)

### 🔍 DIY challenge

1. What proportion of watches recorded **no insects of any kind**?
2. Find the watches at `Weald` in `August` that saw more than two honeybees. How many are there?
3. Use `~` to find every watch that was **not** on a community plot, and check your answer against `df["Community"].value_counts()`.

> 💡 Give your mask a name on its own line. `df[df[(df["a"] > 1) & (df["b"] < 2)]]` is where readable code goes to die. If you see `The truth value of a Series is ambiguous`, you used `and` where you meant `&`.

In [ ]:
# TODO - your turn

## Skill 3 — `.isin()` and `.between()`

Once you are chaining three `|` conditions together, there is a shorter way.

```python
df[df["Crop_Generic"].isin(["Squash", "Courgette", "Cucumber"])]
df[df["Total_Flowers"].between(50, 100)]
```

`.isin()` takes a list and asks "is this row's value one of these?". `.between()` is inclusive at both ends. Both return an ordinary mask, so they combine with `&` and `|` like anything else.

In [ ]:
# DEMO - the cucurbits, in one readable line
cucurbits = ["Squash", "Courgette", "Cucumber"]
c = df[df["Crop_Generic"].isin(cucurbits)]

print(len(c), "watches on cucurbits")
print(c["Crop_Generic"].value_counts())

print()
mid = df[df["Total_Flowers"].between(50, 100)]
print(len(mid), "watches with 50-100 flowers in view")

### 🔍 DIY challenge

1. Pull out every watch from the three busiest allotments. Get the names from `.value_counts().head(3).index`.
2. Find the watches in the **summer months** — June, July and August — with `.isin()`.
3. Combine them: summer watches, at those three allotments, that saw at least one bumblebee. How many?

> 💡 `.value_counts()` returns a Series whose **index** is the names. `.index` hands that straight to `.isin()`.

In [ ]:
# TODO - your turn

## Skill 4 — Selecting on time

`Date` arrives as **text**. Sorting it happens to work, because these dates are written year-first. Everything else fails: you cannot select a month, or ask how far apart two watches were.

```python
df["Date"] = pd.to_datetime(df["Date"])
```

Once a column is `datetime64` it grows a `.dt` accessor — the date version of the `.str` that fixed the header in notebook 1. And the moment you have real dates, you can **check the `Month` column somebody typed by hand** against one you derive yourself.

In [ ]:
# DEMO - one line, and the column wakes up
df["Date"] = pd.to_datetime(df["Date"])
print("dtype now:", df["Date"].dtype)

print()
print("first watch:", df["Date"].min().date())
print("last watch: ", df["Date"].max().date())
print("survey days:", df["Date"].nunique())
print("span:       ", (df["Date"].max() - df["Date"].min()).days, "days")

In [ ]:
# DEMO - trust, but verify
derived = df["Date"].dt.month_name()
mismatch = df[derived != df["Month"]]

print(len(mismatch), "rows where the Month column disagrees with the Date")
mismatch[["Date", "Month", "Allotment", "Plot", "Crop_Generic"]]

Six watches at Moulsecoomb, all on **1 September 2017**, all labelled August. Somebody filled in the month when the survey round began and did not change it when the round ran into a new month. Completely ordinary, and in every hand-entered dataset you will ever touch.

**The lesson generalises:** when a value can be *derived* and also *typed by hand*, derive it.

In [ ]:
# DEMO - derive it, and the disagreement is gone by construction
df["Month"] = df["Date"].dt.month_name()
df["MonthNum"] = df["Date"].dt.month

print((df["Date"].dt.month_name() != df["Month"]).sum(), "mismatches now")
print()
print(df["Month"].value_counts())

There is one more trap, and it will appear on the x-axis of your first seasonal figure:

```python
sorted(df["Month"].unique())
# ['April', 'August', 'July', 'June', 'May', 'September']
```

That is what "sorted" means for text. Tell pandas the order once, with an **ordered Categorical** — a text column that knows what comes after what. Sorting then does the right thing everywhere, including inside `groupby` and on a plot axis. Better still, it makes the column **comparable**, so `df["Month"] > "June"` becomes a legal selection.

In [ ]:
# DEMO - fix 2, the one worth learning
month_order = ["April", "May", "June", "July", "August", "September"]
df["Month"] = pd.Categorical(df["Month"], categories=month_order, ordered=True)

print(df["Month"].dtype)
print()
print("now it sorts like a season:")
print(df["Month"].value_counts().sort_index())

In [ ]:
# DEMO - comparisons that only work because the column is ordered
late = df[df["Month"] > "June"]
print(len(late), "watches in July, August and September")
print(late["Month"].value_counts().sort_index())

### 🔍 DIY challenge

1. Does the `Year` column agree with `.dt.year` everywhere? Check it the way you just checked `Month`.
2. Select the **late summer** watches — July onwards — and count how many bumblebees they saw.
3. Add `DayOfYear` with `.dt.dayofyear`, and select every watch from the first half of the year.

> 💡 When a value can be *derived* and also *typed by hand*, derive it. A derived column cannot disagree with its source.

In [ ]:
# TODO - your turn

## Skill 5 — Adding a column 🏆

Everything so far has taken things out of the table. Now put something in. Assigning to a name that does not exist yet **creates** the column:

```python
df["All_Insects"] = df[INSECTS].sum(axis=1)
```

This is where pandas earns its keep. The arithmetic runs over all 1,620 rows at once, and the result lines up row for row — no loop, no index bookkeeping.

The column worth building here is a **rate**: visits *per flower*. A watch that saw 5 bees on 10 flowers is a very different thing from one that saw 5 bees on 2,000 flowers, and the raw count cannot tell them apart.

Once the column exists you will want the top of it. `.sort_values()` reorders the whole table and hands back a **sorted copy**; `.nlargest(n, column)` is shorter when you only want the extremes.

In [ ]:
# DEMO - two new columns, and the difference between them
df["All_Insects"] = df[INSECTS].sum(axis=1)
df["Visits_Per_Flower"] = df["All_Insects"] / df["Total_Flowers"]

look = df.nlargest(5, "All_Insects")
print("--- busiest by raw count ---")
print(look[["Crop_Generic", "Total_Flowers", "All_Insects", "Visits_Per_Flower"]])

print()
print("--- busiest by visits per flower ---")
print(df.nlargest(5, "Visits_Per_Flower")[
    ["Crop_Generic", "Total_Flowers", "All_Insects", "Visits_Per_Flower"]])

Two completely different top-fives, from the same 1,620 rows. Neither is wrong. They answer different questions, and the job of an analyst is to know which one was asked.

We will come back to this properly in session 3 — it is the central idea of the whole lab.

In [ ]:
# DEMO - which single watch saw the most bumblebees?
df.nlargest(5, "Bumblebees")[
    ["Date", "Allotment", "Crop_Generic", "Total_Flowers", "Bumblebees"]]

### 🏆 DIY finale — build your own view of the data

1. Add `Bee_Visits`: honeybees plus bumblebees plus solitary bees.
2. Build a table `busy` holding only the watches that saw at least one insect **and** had at least 20 flowers in view. How many rows survive?
3. From `busy`, which crop appears most often? Which month?
4. Sort the whole table by `Visits_Per_Flower` and look at the top ten. What do they have in common, and does it make you trust that column less?
5. Is `Visits_Per_Flower` higher on community or individual plots? Compare the means — and keep the answer, because notebook 4 will show you it is the wrong way to ask.

> 💡 Question 4 has a real finding in it. Look at the `Flowers` column of those top ten rows.

In [ ]:
# TODO - your turn

## 🔭 Explore further

- The pandas indexing guide — `.loc` vs `.iloc` is worth twenty minutes: <https://pandas.pydata.org/docs/user_guide/indexing.html>
- `df.query("Year == 2018 and Bumblebees > 0")` filters with a string instead. Some people love it; some find it hides the brackets you need to learn.
- **Extra notebook:** `extras/01_cleaning_deeper` — a date index you can slice with `ts.loc["2018-07"]`, and `.resample()` for grouping by calendar period.

## Recap

You can now:

- select by label with `.loc` and by position with `.iloc`, and say which you want
- build a boolean mask, count it with `.sum()`, take a proportion with `.mean()`
- combine conditions with `&`, `|`, `~` — each in its own brackets
- shorten a pile of `|` into `.isin()`, and a range into `.between()`
- convert text to real dates, check a hand-typed column against a derived one, and select on time
- create new columns — including the one that matters, a **rate** rather than a count

**Next:** `03_groupby_basics` — one line that splits the table, summarises each piece, and puts it back together.